In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import joblib

## 加载数据

In [2]:
# 读取Excel文件
df = pd.read_csv("/home/zhanyu/experiment/data-hh/海航系航班销售过程数据202301-202406-新/2023-01-02.csv")

In [3]:
# 查看数据的前几行
print(df.head(60))

          航班日期             A             B             C  \
0   2023-01-02  vbo2dwBOeps=  KNqX4/Q5Noc=           NaN   
1   2023-01-02  vbo2dwBOeps=  KNqX4/Q5Noc=           NaN   
2   2023-01-02  KNqX4/Q5Noc=  vbo2dwBOeps=           NaN   
3   2023-01-02  KNqX4/Q5Noc=  vbo2dwBOeps=           NaN   
4   2023-01-02  KNqX4/Q5Noc=  vbo2dwBOeps=           NaN   
5   2023-01-02  KNqX4/Q5Noc=  vbo2dwBOeps=           NaN   
6   2023-01-02  KNqX4/Q5Noc=  vbo2dwBOeps=           NaN   
7   2023-01-02  KNqX4/Q5Noc=  vbo2dwBOeps=           NaN   
8   2023-01-02  KNqX4/Q5Noc=  vbo2dwBOeps=           NaN   
9   2023-01-02  tKjndGSl9NQ=  Lue5PP9SfQU=           NaN   
10  2023-01-02  Lue5PP9SfQU=  tKjndGSl9NQ=           NaN   
11  2023-01-02  Lue5PP9SfQU=  tKjndGSl9NQ=           NaN   
12  2023-01-02  Lue5PP9SfQU=  tKjndGSl9NQ=           NaN   
13  2023-01-02  Lue5PP9SfQU=  tKjndGSl9NQ=           NaN   
14  2023-01-02  Lue5PP9SfQU=  tKjndGSl9NQ=           NaN   
15  2023-01-02  Lue5PP9SfQU=  tKjndGSl9N

In [4]:
# 统计 flightNo 字段中不同取值的数量和出现次数
flight_counts = df['航班号'].value_counts()

# 查看结果
print(flight_counts)

航班号
xqmzDfE8pZs=    168
MGhRGDWjWTA=    160
8mSRwscwKZM=    160
rHKBukrqA5c=    160
RYVGsHdC/4o=    155
               ... 
ofQp7niPbjc=      1
ACmWWI4dJJI=      1
ccIOSUC6Rn0=      1
okYnGw3gtAs=      1
nHQm15ff9Y4=      1
Name: count, Length: 5840, dtype: int64


In [5]:
# 筛选出航班号为 'A4tgQv5cqkc=' 的行
filtered_df = df[df['航班号'] == 'A4tgQv5cqkc=']

# 查看筛选后的结果
print(filtered_df)

# 假设 df 是你的 DataFrame
filtered_df.to_csv('A4tgQv5cqkc=_1-2.csv', index=False, encoding='utf-8-sig')

             航班日期             A             B             C  \
38871  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38872  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38873  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38874  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38875  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
...           ...           ...           ...           ...   
62505  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62506  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62507  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62508  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62509  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   

                              航段           航班号  距离天数   人数  
38871  xIqe+QQIQXQ=-t+kFcRdzL6o=  A4tgQv5cqkc=     6   69  
38872  xIqe+QQIQXQ=-t+kFcRdzL6o=  A4tgQv5cqkc=     5   73  
38873  xIqe+QQIQXQ=-t+kFcRdzL6o=  A4tgQv5cqkc=     4   69  
388

In [6]:
# 按 'A', 'B', 'C', '航段' 分组并统计出现次数
comb_counts = filtered_df.groupby(['A', 'B', 'C', '航段']).size()

# 将结果查看为 Series 格式
print(comb_counts)

A             B             C             航段                       
xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=  t+kFcRdzL6o=-/M8gHzjxpNU=    38
                                          xIqe+QQIQXQ=-/M8gHzjxpNU=    38
                                          xIqe+QQIQXQ=-t+kFcRdzL6o=    38
dtype: int64


In [7]:
# 统计 flightNo 字段中不同取值的数量和出现次数
flight_counts = filtered_df['距离天数'].value_counts()
# 查看结果
print(flight_counts)

距离天数
 6     6
 4     6
 3     6
 2     6
 1     6
 0     6
 5     6
 12    3
 15    3
 14    3
 13    3
 9     3
 11    3
 10    3
 17    3
 8     3
 7     3
 16    3
 21    3
 18    3
 19    3
 20    3
 23    3
 24    3
 25    3
 26    3
 27    3
 28    3
 29    3
 22    3
-1     3
Name: count, dtype: int64


In [14]:
print(filtered_df)

             航班日期             A             B             C  \
38871  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38872  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38873  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38874  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38875  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
...           ...           ...           ...           ...   
62505  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62506  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62507  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62508  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62509  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   

                              航段           航班号  距离天数   人数  
38871  xIqe+QQIQXQ=-t+kFcRdzL6o=  A4tgQv5cqkc=     6   69  
38872  xIqe+QQIQXQ=-t+kFcRdzL6o=  A4tgQv5cqkc=     5   73  
38873  xIqe+QQIQXQ=-t+kFcRdzL6o=  A4tgQv5cqkc=     4   69  
388

In [16]:
 # 对筛选后的数据按 '航班日期', 'A', 'B', 'C', '航段', '航班号', '距离天数' 分组，并保留人数最大的行
filtered_df = filtered_df.loc[filtered_df.groupby(['航班日期', 'A', 'B', 'C', '航段', '航班号', '距离天数'])['人数'].idxmax()]

In [17]:
print(filtered_df)

             航班日期             A             B             C  \
62509  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38891  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38890  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38889  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
38888  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
...           ...           ...           ...           ...   
62421  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62420  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62419  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62418  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
62417  2023-01-02  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   

                              航段           航班号  距离天数   人数  
62509  t+kFcRdzL6o=-/M8gHzjxpNU=  A4tgQv5cqkc=    -1  144  
38891  t+kFcRdzL6o=-/M8gHzjxpNU=  A4tgQv5cqkc=     0  144  
38890  t+kFcRdzL6o=-/M8gHzjxpNU=  A4tgQv5cqkc=     1  117  
388